# FreshCart ELT on Snowflake

This notebook loads the sample website/mobile order events into `BRONZE.ORDERS_RAW`, applies the SQL transforms, and runs quality checks. It is designed to run from the repository root after setting the required `SNOWFLAKE_*` environment variables. The SQL and points policy are a self-contained implementation; compare the policy with the class FreshCart notebook before submitting results.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "elt" / "sql").is_dir() and (ROOT.parent / "elt" / "sql").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from elt.sample_orders import MOCK_ORDERS
from elt.snowflake_io import connect_from_env, execute_transform_sql, load_file, write_landing
from elt.quality_checks import quality_checks

SQL_DIR = ROOT / "elt" / "sql"
connection = connect_from_env()
print(f"Connected to Snowflake database={connection.database}, schema={connection.schema}")

In [ ]:
execute_transform_sql(connection, SQL_DIR / "00_setup.sql")
landing_file = write_landing(MOCK_ORDERS)
load_file(connection, landing_file)
print(f"Loaded {len(MOCK_ORDERS)} mock records from {landing_file}")

In [ ]:
for sql_name in (
    "SQL_10_ORDERS_CLEAN.sql",
    "SQL_20_ORDERS_DEDUP.sql",
    "SQL_30_CUSTOMERS_SCD2.sql",
    "SQL_40_GOLD_MERGE.sql",
):
    execute_transform_sql(connection, SQL_DIR / sql_name)
    print(f"Executed {sql_name}")

In [ ]:
count_queries = {
    "bronze.orders_raw": "SELECT COUNT(*) FROM bronze.orders_raw",
    "silver.orders_clean": "SELECT COUNT(*) FROM silver.orders_clean",
    "silver.orders_quarantine": "SELECT COUNT(*) FROM silver.orders_quarantine",
    "silver.orders_dedup": "SELECT COUNT(*) FROM silver.orders_dedup",
    "gold.fct_orders": "SELECT COUNT(*) FROM gold.fct_orders",
}
with connection.cursor() as cursor:
    for label, query in count_queries.items():
        cursor.execute(query)
        print(f"{label}: {cursor.fetchone()[0]}")
    cursor.execute("SELECT SUM(amount_inr), SUM(loyalty_points) FROM gold.fct_orders")
    revenue, points = cursor.fetchone()
    print(f"Gold revenue INR: {revenue}; loyalty points: {points}")
    cursor.execute("SELECT order_id, published_at FROM gold.fct_orders")
    baseline_published_at = dict(cursor.fetchall())

In [ ]:
with connection.cursor() as cursor:
    cursor.execute(
        "SELECT raw_payload FROM bronze.orders_raw "
        "WHERE COALESCE(raw_payload:order_id::STRING, raw_payload:orderId::STRING) = 'ORD-1003' "
        "ORDER BY loaded_at"
    )
    print("ORD-1003 bronze records:")
    for row in cursor.fetchall():
        print(row[0])

    cursor.execute(
        "SELECT order_id, amount, currency, channel, reasons "
        "FROM silver.orders_checked WHERE order_id = 'ORD-1003' ORDER BY updated_at"
    )
    print("ORD-1003 normalized records:")
    for row in cursor.fetchall():
        print(row)

    cursor.execute(
        "SELECT order_id, reasons FROM silver.orders_checked WHERE order_id = 'ORD-1006'"
    )
    print("ORD-1006 quarantine reasons:", cursor.fetchall())

check_results = quality_checks(connection)
assert all(status == "PASS" for _, _, status in check_results)

## B3: Currency validation

The SQL transform includes the unsupported-currency rule. The next local comparison shows how the ORD-1006 reasons array changes while the record remains quarantined; the Snowflake quarantine `MERGE` is keyed by `record_hash`.

In [ ]:
from elt.local_model import clean_records

_, quarantine_before = clean_records(MOCK_ORDERS, validate_currency=False)
_, quarantine_after = clean_records(MOCK_ORDERS, validate_currency=True)
reason_before = next(row["reasons"] for row in quarantine_before if row["order_id"] == "ORD-1006")
reason_after = next(row["reasons"] for row in quarantine_after if row["order_id"] == "ORD-1006")
print("ORD-1006 reasons before currency rule:", reason_before)
print("ORD-1006 reasons after currency rule:", reason_after)
print(f"Quarantine rows before/after: {len(quarantine_before)} / {len(quarantine_after)}")

## B4: CRM tier event

Before running the next cell, write and save your own prediction table in the report. The cell adds Arjun's Gold event effective 2026-08-15, rebuilds SCD2 and gold, then prints the actual CUST-02 versions and loyalty total.

In [ ]:
with connection.cursor() as cursor:
    cursor.execute("""
        MERGE INTO REF.MOCK_CRM_TIER_EVENTS target
        USING (SELECT 'CUST-02' AS customer_id, 'arjun.k@yahoo.in' AS customer_email,
                      'Gold' AS tier, TO_DATE('2026-08-15') AS effective_from) source
        ON target.customer_id = source.customer_id
           AND target.effective_from = source.effective_from
        WHEN NOT MATCHED THEN INSERT (customer_id, customer_email, tier, effective_from)
        VALUES (source.customer_id, source.customer_email, source.tier, source.effective_from)
    """)

execute_transform_sql(connection, SQL_DIR / "SQL_30_CUSTOMERS_SCD2.sql")
execute_transform_sql(connection, SQL_DIR / "SQL_40_GOLD_MERGE.sql")
with connection.cursor() as cursor:
    cursor.execute("SELECT customer_id, tier, effective_from, effective_to FROM dim.customers_scd2 WHERE customer_id = 'CUST-02' ORDER BY effective_from")
    print("CUST-02 SCD2 versions:", cursor.fetchall())
    cursor.execute("SELECT tier, loyalty_points FROM gold.fct_orders WHERE order_id = 'ORD-1002'")
    print("ORD-1002 after CRM event:", cursor.fetchone())
    cursor.execute("SELECT SUM(loyalty_points) FROM gold.fct_orders")
    print("Gold points after CRM event:", cursor.fetchone()[0])
    cursor.execute("SELECT order_id, published_at FROM gold.fct_orders")
    after_event = dict(cursor.fetchall())

changed_published_at = sum(
    before != after_event[order_id]
    for order_id, before in baseline_published_at.items()
    if order_id in after_event
)
print(f"Gold rows with changed published_at: {changed_published_at}")

In [ ]:
connection.close()
print("Snowflake connection closed")